# AMC 2026 – Round 3 · Notebook 1: the Bloomberg data for the macro strategy
Financial Modeling (USI) · TA session 7 · *run it after refreshing `r7_bloomberg.xlsx` on the terminal*

**What this notebook does.** It reads the Bloomberg data, cleans it, checks it, and saves CSV files for Notebook 2. Nothing is analysed here, it is only data.

### Input files

| File | Sheet | What it contains |
|---|---|---|
| `Data/r7_bloomberg_values.xlsx` | **Macro** | ISM manufacturing PMI, US CPI y/y, 3-month T-bill, Fed funds (monthly, from 1990) |
| | **Assets** | total-return indices of the SAA asset classes (from 1990) |
| | **Countries** | total return index of 18 iShares MSCI country ETFs (from 2001) |
| | **Value** | price-to-book of the MSCI index of each country |
| | **Live**, **Rates** | last price of the country ETFs; today's policy rates |
| `Data/DataPython.xlsx` | | the carry data of the *Efficiently Inefficient* exercise (9 currencies, 1980–2014) |
| `Data/r3_*.csv` | | the output of TA session 6 (the Round 3 portfolio with the quant sleeve) |

### Output files

| CSV | Contents | Used in Notebook 2 for |
|---|---|---|
| `r7_macro.csv` | PMI, CPI y/y, T-bill, Fed funds | §2 regimes |
| `r7_assets.csv` | asset-class indices, named after their SAA ETF | §2 regimes |
| `r7_rf_long.csv` | monthly T-bill rate since 1990 | §2, §3 excess returns |
| `r7_countries.csv`, `r7_pb.csv` | country ETF prices and P/B | §3 country value & momentum |
| `r7_live.csv`, `r7_rates.csv` | today's prices and policy rates | §3, §4, §6 |
| `r7_carry_ret.csv`, `r7_carry_rates.csv` | FX excess returns and interest rates | §4 carry |

In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.float_format", "{:.4f}".format)

DATA = "Data/"                                     # subfolder with the Bloomberg files
FILE = DATA + "r7_bloomberg_values.xlsx"           # r7_bloomberg.xlsx after the refresh, saved as values

# the 18 country ETFs and the MSCI index used for their P/B
COUNTRIES = {"EWA": ("Australia", "MXAU"), "EWC": ("Canada", "MXCA"), "EWG": ("Germany", "MXDE"),
             "EWJ": ("Japan", "MXJP"), "EWU": ("United Kingdom", "MXGB"), "EWQ": ("France", "MXFR"),
             "EWI": ("Italy", "MXIT"), "EWP": ("Spain", "MXES"), "EWL": ("Switzerland", "MXCH"),
             "EWN": ("Netherlands", "MXNL"), "EWD": ("Sweden", "MXSE"), "EWH": ("Hong Kong", "MXHK"),
             "EWS": ("Singapore", "MXSG"), "EWY": ("South Korea", "MXKR"), "EWT": ("Taiwan", "MXTW"),
             "EWZ": ("Brazil", "MXBR"), "EWW": ("Mexico", "MXMX"), "SPY": ("United States", "MXUS"),
             # Matterhorn additions
             "MCHI": ("China", "MXCN"), "INDA": ("India", "MXIN"), "EZA": ("South Africa", "MXZA")}
# long-history index -> the SAA ETF that implements the same asset class
ASSETS = {"SPXT": "SPY", "LUATTRUU": "IEF", "LBUTTRUU": "TIP", "LUACTRUU": "LQD", "BCOMTR": "DBC", "XAU": "GLD",
          # Matterhorn additions: long-history proxies of our SAA (TIP, GLD, BCI use LBUTTRUU, XAU, BCOMTR)
          "RU30INTR": "VTI", "NDDUWXUS": "VEA", "NDUEEGF": "IEMG", "LBUSTRUU": "BND", "LEGATRUH": "BNDX"}
MACRO  = {"NAPMPMI": "PMI", "CPI YOY": "CPI_yoy", "USGG3M": "T-bill 3M", "FDTR": "Fed funds"}

def to_dates(d):
    # Excel dates can arrive as dates or as serial numbers (days since 1899-12-30)
    if pd.api.types.is_datetime64_any_dtype(d):
        return d
    num = pd.to_numeric(d, errors="coerce")
    if num.notna().sum() > d.notna().sum() / 2:
        return pd.to_datetime(num, unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(d, errors="coerce", format="mixed")

def read_bdh(sheet):
    # same reader as TA session 6: row 1 = ticker, then two columns (date, value) per ticker
    raw = pd.read_excel(FILE, sheet_name=sheet, header=None)
    out = {}
    for j in range(0, raw.shape[1] - 1, 2):
        t = raw.iat[0, j]
        if not isinstance(t, str) or not t.strip():
            continue
        s = pd.Series(pd.to_numeric(raw.iloc[1:, j + 1], errors="coerce").values,
                      index=to_dates(raw.iloc[1:, j]).values).dropna()          # '#N/A ...' rows are dropped
        s = s[s.index.notna()]
        if len(s):
            name = t.replace(" US Equity", "").replace(" Equity", "").replace(" Index", "").replace(" Curncy", "")
            out[name] = s
        else:
            print(f"⚠️ {sheet}: no data for {t} -> check the ticker on the terminal")
    df = pd.DataFrame(out)
    df.index = pd.DatetimeIndex(df.index) + pd.offsets.MonthEnd(0)              # label = calendar month-end
    return df.groupby(level=0).last().sort_index()

def check(df, name):
    # one line per series: first date, last date, months, missing values inside the history
    first, last = df.apply(pd.Series.first_valid_index), df.apply(pd.Series.last_valid_index)
    holes = {c: int(df.loc[first[c]:last[c], c].isna().sum()) for c in df}
    print(f"{name}: {df.shape[1]} series, {df.index[0]:%b %Y} -> {df.index[-1]:%b %Y}")
    return pd.DataFrame({"first": first.dt.strftime("%Y-%m"), "last": last.dt.strftime("%Y-%m"),
                         "months": df.notna().sum(), "holes": pd.Series(holes)})

## 1. Macro data
- **PMI** (`NAPMPMI Index`): survey of purchasing managers; above 50 = manufacturing expands. It is our **growth** signal.
- **CPI y/y** (`CPI YOY Index`, %): our **inflation** signal.
- **T-bill 3M** (`USGG3M Index`, % per year) becomes the monthly risk-free rate: $r_f = \text{yield}/100/12$. We use it instead of `GB1M` because it starts earlier.

In [2]:
macro = read_bdh("Macro").rename(columns=MACRO)
rf_long = (macro["T-bill 3M"] / 100 / 12).rename("rf")

display(check(macro, "Macro"))
print(f"Last values: PMI {macro['PMI'].dropna().iloc[-1]:.1f} | CPI y/y {macro['CPI_yoy'].dropna().iloc[-1]:.1f}% | "
      f"T-bill {macro['T-bill 3M'].dropna().iloc[-1]:.2f}% | Fed funds {macro['Fed funds'].dropna().iloc[-1]:.2f}%")
# sanity: the PMI lives between ~30 and ~65, CPI y/y between ~-2% and ~10% since 1990
macro.describe().T[["count", "mean", "min", "max"]]

Macro: 4 series, Dec 1989 -> Oct 2026


,first,last,months,holes
PMI,1989-12,2026-10,443,0
CPI_yoy,1989-12,2026-10,443,0
T-bill 3M,1990-01,2026-10,442,0
Fed funds,1990-01,2026-10,442,0


Last values: PMI 55.5 | CPI y/y 3.7% | T-bill 4.08% | Fed funds 4.00%


,count,mean,min,max
PMI,443.0000,52.2284,34.5000,63.8000
CPI_yoy,443.0000,2.7230,-2.1000,9.1000
T-bill 3M,442.0000,2.7860,-0.0203,8.0440
Fed funds,442.0000,2.9553,0.2500,8.2500


## 2. Asset-class indices
Total-return indices with a long history, so that we have many months in each macro regime. Each one is renamed after the **SAA ETF** that invests in the same asset class (the ETFs themselves start only in 2002–2006).

In [3]:
assets = read_bdh("Assets").rename(columns=ASSETS)
display(check(assets, "Assets"))

R_assets = assets.pct_change(fill_method=None)
big = (R_assets.abs() > 0.3).sum()                                  # a monthly move > 30% is suspicious for these indices
print("Months with |return| > 30%:", big[big > 0].to_dict() if big.any() else "none")
(R_assets.describe().T[["count", "mean", "std", "min", "max"]])

Assets: 11 series, Jan 1990 -> Oct 2026

,first,last,months,holes
SPY,1990-01,2026-10,442,0
IEF,1990-01,2026-10,442,0
TIP,1997-03,2026-10,356,0
LQD,1990-01,2026-10,442,0
DBC,1990-01,2026-10,442,0
GLD,1990-01,2026-10,442,0
VTI,1990-01,2026-10,442,0
VEA,1990-01,2026-10,442,0
IEMG,1998-12,2026-10,335,0
BND,1990-01,2026-10,442,0


Months with |return| > 30%: none


,count,mean,std,min,max
SPY,441.0000,0.0097,0.0424,-0.1680,0.1282
IEF,441.0000,0.0039,0.0133,-0.0439,0.0531
TIP,355.0000,0.0039,0.0158,-0.0869,0.0584
LQD,441.0000,0.0048,0.0168,-0.0777,0.0680
DBC,441.0000,0.0042,0.0427,-0.2128,0.1380
GLD,441.0000,0.0061,0.0452,-0.1689,0.1685
VTI,441.0000,0.0097,0.0433,-0.1774,0.1324
VEA,441.0000,0.0057,0.0475,-0.2080,0.1534
IEMG,334.0000,0.0088,0.0605,-0.2737,0.1709
BND,441.0000,0.0041,0.0119,-0.0432,0.0453


## 3. Country ETFs and their valuation
- **Prices:** total return index of the iShares MSCI country ETFs, in **USD** (so the return includes the currency).
- **Value:** price-to-book of the MSCI index of the same country. We need it every month back to 2001, for the backtest.

In [4]:
countries = read_bdh("Countries")
pb = read_bdh("Value").rename(columns={v[1]: k for k, v in COUNTRIES.items()})   # MSCI ticker -> ETF ticker
roe = read_bdh("ROE").rename(columns={v[1]: k for k, v in COUNTRIES.items()})  # Matterhorn: ROE of the MSCI index, for the P/B adjusted for ROE

display(check(countries, "Country ETFs").join(check(pb, "P/B")[["first", "months"]], rsuffix=" P/B"))

R_c = countries.pct_change(fill_method=None)
print(f"Months with |return| > 40%: {int((R_c.abs() > 0.4).sum().sum())}  |  "
      f"P/B <= 0 or > 20: {int(((pb <= 0) | (pb > 20)).sum().sum())}")
missing = [k for k in COUNTRIES if k not in pb]
if missing:
    print("⚠️ no P/B for", missing, "-> notebook 2 uses only momentum for these countries")

Country ETFs: 21 series, Jan 2001 -> Oct 2026
P/B: 21 series, Jan 2001 -> Oct 2026


,first,last,months,holes,first P/B,months P/B
EWA,2001-01,2026-10,310,0,2001-01,310
EWC,2001-01,2026-10,310,0,2001-01,310
EWG,2001-01,2026-10,310,0,2001-01,310
EWJ,2001-01,2026-10,310,0,2001-01,310
EWU,2001-01,2026-10,310,0,2001-01,310
EWQ,2001-01,2026-10,310,0,2001-01,310
EWI,2001-01,2026-10,310,0,2001-01,310
EWP,2001-01,2026-10,310,0,2001-01,310
EWL,2001-01,2026-10,310,0,2001-01,310
EWN,2001-01,2026-10,310,0,2001-01,310


Months with |return| > 40%: 0  |  P/B <= 0 or > 20: 0


## 4. Today's prices and policy rates
- `Live`: last price of the country ETFs (the closes of **Friday 27 November** in the challenge).
- `Rates`: the policy rate of each carry currency today, in % per year. ⚠️ If a ticker returned `#N/A`, type the rate by hand in `RATES_BY_HAND` (source: the central bank's website).

In [5]:
live = pd.read_excel(FILE, sheet_name="Live")
live.columns = ["ticker", "name", "px_last"]
live["ticker"] = live["ticker"].str.replace(" US Equity", "")
live["px_last"] = pd.to_numeric(live["px_last"], errors="coerce")

rates = pd.read_excel(FILE, sheet_name="Rates").iloc[:, :3]
rates.columns = ["currency", "ticker", "rate"]
rates["rate"] = pd.to_numeric(rates["rate"], errors="coerce")
RATES_BY_HAND = {}                                   # e.g. {"Switzerland": 0.0} if the BDP did not work
for c, r in RATES_BY_HAND.items():
    rates.loc[rates["currency"] == c, "rate"] = r

print(f"Live prices: {live['px_last'].notna().sum()} of {len(live)} | policy rates: {rates['rate'].notna().sum()} of {len(rates)}")
display(rates)
live

Live prices: 36 of 36 | policy rates: 12 of 13


,currency,ticker,rate
0,US,FDTR Index,4.0000
1,Australia,RBATCTR Index,4.8500
2,Germany,EUORDEPO Index,2.5000
3,Canada,CABROVER Index,2.5000
4,Japan,BOJDTR Index,1.2500
5,NewZealand,NZOCRS Index,2.7500
6,Sweden,SWRRATEI Index,2.0000
7,Switzerland,SZLTTR Index,-0.7500
8,UK,UKBRBASE Index,4.0000
9,Brazil,BZSTSETA Index,13.7500


,ticker,name,px_last
0,EWA,ISHARES MSCI AUSTRALIA ETF,27.8800
1,EWC,ISHARES MSCI CANADA ETF,57.7700
2,EWG,ISHARES MSCI GERMANY ETF,40.5200
3,EWJ,ISHARES MSCI JAPAN ETF,89.0700
4,EWU,ISHARES MSCI UNITED KINGDOM,46.8400
5,EWQ,ISHARES MSCI FRANCE ETF,42.3700
6,EWI,ISHARES MSCI ITALY ETF,55.8000
7,EWP,ISHARES MSCI SPAIN ETF,57.5600
8,EWL,ISHARES MSCI SWITZERLAND ETF,64.0300
9,EWN,ISHARES MSCI NETHERLANDS ETF,60.9600


## 5. The carry data (Pedersen, *Efficiently Inefficient*, exercise 11)
Two sheets, 9 currencies, monthly, 1980–2014:
- `InterestRate`: the annual short rate of each country at the date;
- `ExcessReturn`: the return of investing USD 1 in the foreign currency (deposit + currency move) **over the US rate**, for the month **ending** at the date.

In [6]:
carry_ret   = pd.read_excel(DATA + "DataPython.xlsx", sheet_name="ExcessReturn", index_col=0).iloc[1:]   # first row is empty
carry_rates = pd.read_excel(DATA + "DataPython.xlsx", sheet_name="InterestRate", index_col=0)
print(f"{carry_ret.index[0]:%b %Y} -> {carry_ret.index[-1]:%b %Y} | {len(carry_ret)} months | "
      f"missing: {carry_ret.isna().sum().sum()} returns, {carry_rates.isna().sum().sum()} rates")
carry_ret.describe().T[["mean", "std", "min", "max"]]

Jan 1980 -> Sep 2014 | 417 months | missing: 0 returns, 0 rates


,mean,std,min,max
Australia,0.0022,0.0332,-0.1625,0.0956
Germany,-0.0001,0.0323,-0.1024,0.0957
Canada,0.0009,0.0199,-0.1270,0.0839
Japan,0.0001,0.0334,-0.1057,0.1642
NewZealand,0.0024,0.0361,-0.2242,0.1431
Sweden,-0.0000,0.0326,-0.1568,0.0945
Switzerland,-0.0001,0.0350,-0.1132,0.1378
UK,0.0009,0.0298,-0.1148,0.1405
US,0.0000,0.0000,0.0000,0.0000


In [7]:
macro.to_csv("r7_macro.csv");           assets.to_csv("r7_assets.csv");     rf_long.to_csv("r7_rf_long.csv")
countries.to_csv("r7_countries.csv");   pb.to_csv("r7_pb.csv");             roe.to_csv("r7_roe.csv")
live.to_csv("r7_live.csv", index=False); rates.to_csv("r7_rates.csv", index=False)
carry_ret.to_csv("r7_carry_ret.csv");   carry_rates.to_csv("r7_carry_rates.csv")
print("saved: r7_macro, r7_assets, r7_rf_long, r7_countries, r7_pb, r7_live, r7_rates, r7_carry_ret, r7_carry_rates (.csv)")

# Matterhorn additions: carry history after 2014 and extra macro series
for sheet, out in [("CarryRates", "r7_carry_rates_bbg.csv"), ("CarryFX", "r7_carry_fx_bbg.csv"), ("Extra", "r7_extra.csv")]:
    df = read_bdh(sheet)
    display(check(df, sheet))
    df.to_csv(out)

saved: r7_macro, r7_assets, r7_rf_long, r7_countries, r7_pb, r7_live, r7_rates, r7_carry_ret, r7_carry_rates (.csv)


CarryRates: 12 series, Jan 2001 -> Oct 2026


,first,last,months,holes
FDTR,2001-01,2026-10,310,0
RBATCTR,2001-02,2026-10,309,0
EUORDEPO,2001-01,2026-10,310,0
CABROVER,2001-01,2026-10,310,0
BOJDTR,2001-01,2026-10,310,0
NZOCRS,2001-01,2026-10,310,0
SWRRATEI,2001-01,2026-10,310,0
SZLTTR,2001-01,2026-10,310,0
UKBRBASE,2001-01,2026-10,310,0
BZSTSETA,2001-01,2026-10,310,0


CarryFX: 12 series, Jan 2001 -> Oct 2026


,first,last,months,holes
EURUSD,2001-01,2026-10,310,0
USDJPY,2001-01,2026-10,310,0
GBPUSD,2001-01,2026-10,310,0
USDCAD,2001-01,2026-10,310,0
AUDUSD,2001-01,2026-10,310,0
NZDUSD,2001-01,2026-10,310,0
USDSEK,2001-01,2026-10,310,0
USDCHF,2001-01,2026-10,310,0
USDBRL,2001-01,2026-10,310,0
USDMXN,2001-01,2026-10,310,0


Extra: 10 series, Dec 1989 -> Oct 2026


,first,last,months,holes
CFNAI,1989-12,2026-10,443,0
USURTOT,1989-12,2026-10,443,0
CPI XYOY,1989-12,2026-10,443,0
USGG2YR,1990-01,2026-10,442,0
USGG10YR,1990-01,2026-10,442,0
USGGBE10,1998-08,2026-10,339,0
USGGT10Y,1997-01,2026-10,358,0
CO1 Comdty,1990-01,2026-10,442,0
CL1 Comdty,1990-01,2026-10,442,0
HG1 Comdty,1990-01,2026-10,442,0
